# Train + Eval Faster R-CNN — Car Damage Detection

Notebook này train **Faster R-CNN** bằng PyTorch/Torchvision với cấu hình giống YOLOv8s:

- `epochs = 30`
- `batch = 16`
- `imgsz = 640`
- eval trên cùng tập `val` trong `data.yaml`
- tính COCO metrics bằng `torchmetrics`: `mAP@50:95`, `mAP@50`, `mAP@75`, `mAR`

> Lưu ý: trong thực tế người ta thường dùng **Faster R-CNN** thay vì Fast R-CNN gốc, vì Faster R-CNN có RPN tạo proposal end-to-end. Notebook này dùng `fasterrcnn_resnet50_fpn`.

> `batch = 16` với Faster R-CNN có thể rất tốn VRAM. Notebook giữ đúng cấu hình bạn yêu cầu; nếu bị CUDA OOM thì giảm `BATCH` xuống `2`, `4` hoặc `8`.


In [1]:
# Nếu chạy trên Google Colab, mount Google Drive để lưu kết quả
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!unzip "/content/drive/MyDrive/coco_damage_car_yolov8.zip" -d "/content/data"

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  inflating: /content/data/coco_damage_car_yolov8/train/images/003120_jpg.rf.Dqo3IY3vqjeG5Laq1jRZ.jpg  
  inflating: /content/data/coco_damage_car_yolov8/train/images/003121_jpg.rf.gook4XJCLIJhUAFFgUnM.jpg  
  inflating: /content/data/coco_damage_car_yolov8/train/images/003122_jpg.rf.Clplyzl40NlRPlPInKwV.jpg  
  inflating: /content/data/coco_damage_car_yolov8/train/images/003123_jpg.rf.R3CR81AtaL9UcP3BgGbt.jpg  
  inflating: /content/data/coco_damage_car_yolov8/train/images/003124_jpg.rf.WSZfqewyivwLqAZ9NVCd.jpg  
  inflating: /content/data/coco_damage_car_yolov8/train/images/003125_jpg.rf.xGGGhMrDFmWZe1E60hbX.jpg  
  inflating: /content/data/coco_damage_car_yolov8/train/images/003126_jpg.rf.MZWDtqSxCVhhr19QSBfG.jpg  
  inflating: /content/data/coco_damage_car_yolov8/train/images/003127_jpg.rf.fNH8ysJrDGAEbkqkHeo0.jpg  
  inflating: /content/data/coco_damage_car_yolov8/train/images/003128_jpg.rf.niGlfkFZxXbEeieWz932.jpg  
  infla

In [3]:
!pip install -q torchmetrics pycocotools pyyaml pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.9 MB/s eta 0:00:00


In [4]:
import os
import json
import yaml
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from torchmetrics.detection.mean_ap import MeanAveragePrecision


## 1. Cấu hình train/eval


In [5]:
DATA_YAML = Path("/content/data/coco_damage_car_yolov8/data.yaml")

EPOCHS = 30
BATCH = 16
IMGSZ = 640

PROJECT_DIR = Path("/content/drive/MyDrive/coco_damage_car_yolov8")
RUN_NAME = "fasterrcnn_car_30e_640"
SAVE_DIR = PROJECT_DIR / RUN_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print("Device:", DEVICE)
print("Use AMP:", USE_AMP)
print("Data YAML:", DATA_YAML)
print("Save dir:", SAVE_DIR)


Device: cuda:0
Use AMP: True
Data YAML: /content/data/coco_damage_car_yolov8/data.yaml
Save dir: /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640


## 2. Đọc `data.yaml` và chuẩn bị class names

YOLO dùng class id bắt đầu từ `0`, còn Faster R-CNN của Torchvision dành label `0` cho background. Vì vậy label thật sẽ được cộng thêm `1`.


In [6]:
with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

yaml_dir = DATA_YAML.parent

# Ưu tiên cfg['path'] nếu data.yaml có khai báo root dataset
if "path" in cfg and cfg["path"] is not None:
    dataset_root = Path(cfg["path"])
    if not dataset_root.is_absolute():
        dataset_root = (yaml_dir / dataset_root).resolve()
else:
    dataset_root = yaml_dir.resolve()


def resolve_split_path(p):
    p = Path(p)
    if p.is_absolute():
        return p
    candidate = dataset_root / p
    if candidate.exists():
        return candidate
    return yaml_dir / p

train_source = resolve_split_path(cfg["train"])
val_source = resolve_split_path(cfg["val"])

names = cfg["names"]
if isinstance(names, dict):
    class_names = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
else:
    class_names = list(names)

num_classes = len(class_names) + 1  # +1 background

print("Dataset root:", dataset_root)
print("Train source:", train_source)
print("Val source:", val_source)
print("Classes:", class_names)
print("num_classes including background:", num_classes)


Dataset root: /content/data/coco_damage_car_yolov8
Train source: /content/data/coco_damage_car_yolov8/train/images
Val source: /content/data/coco_damage_car_yolov8/val/images
Classes: ['crack', 'dent', 'glass shatter', 'lamp broken', 'scratch', 'tire flat']
num_classes including background: 7


## 3. Dataset đọc YOLO format và convert bbox sang XYXY

YOLO label format:

```text
class_id x_center y_center width height
```

Torchvision detection target format:

```python
{
    "boxes": Tensor[N, 4],   # [x1, y1, x2, y2]
    "labels": Tensor[N],     # 1..num_classes-1, label 0 là background
}
```


In [7]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def resolve_image_line(line, txt_file):
    p = Path(line.strip())
    if p.is_absolute():
        return p

    candidates = [
        txt_file.parent / p,
        dataset_root / p,
        yaml_dir / p,
    ]
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]


def read_image_paths(source):
    source = Path(source)

    if source.is_file() and source.suffix.lower() == ".txt":
        paths = []
        with open(source, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    paths.append(resolve_image_line(line, source))
        return sorted(paths)

    if source.is_dir():
        paths = []
        for ext in IMG_EXTS:
            paths.extend(source.rglob(f"*{ext}"))
            paths.extend(source.rglob(f"*{ext.upper()}"))
        return sorted(set(paths))

    raise FileNotFoundError(f"Không tìm thấy image source: {source}")


def yolo_label_path_from_image_path(img_path):
    img_path = Path(img_path)
    s = img_path.with_suffix(".txt").as_posix()

    if "/images/" in s:
        return Path(s.replace("/images/", "/labels/"))

    # Fallback: .../train/img.jpg -> .../labels/train/img.txt
    return img_path.parent.parent / "labels" / img_path.parent.name / f"{img_path.stem}.txt"


class YoloFormatDetectionDataset(Dataset):
    def __init__(self, image_source):
        self.image_paths = read_image_paths(image_source)
        if len(self.image_paths) == 0:
            raise RuntimeError(f"Không tìm thấy ảnh trong: {image_source}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label_path = yolo_label_path_from_image_path(img_path)

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue

                    parts = line.split()
                    if len(parts) < 5:
                        continue

                    cls, xc, yc, bw, bh = map(float, parts[:5])

                    x1 = (xc - bw / 2) * w
                    y1 = (yc - bh / 2) * h
                    x2 = (xc + bw / 2) * w
                    y2 = (yc + bh / 2) * h

                    x1 = max(0, min(x1, w - 1))
                    y1 = max(0, min(y1, h - 1))
                    x2 = max(0, min(x2, w))
                    y2 = max(0, min(y2, h))

                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)  # +1 vì label 0 là background

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        if boxes.numel() == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        iscrowd = torch.zeros((boxes.shape[0],), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd,
        }

        img = F.to_tensor(img)
        return img, target


## 4. DataLoader


In [8]:
def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = YoloFormatDetectionDataset(train_source)
val_dataset = YoloFormatDetectionDataset(val_source)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)

print("Train images:", len(train_dataset))
print("Val images  :", len(val_dataset))


Train images: 2800
Val images  : 800


## 5. Tạo model Faster R-CNN


In [9]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights,
    min_size=IMGSZ,
    max_size=IMGSZ,
)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model.to(DEVICE)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005,
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.1,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print(model.__class__.__name__)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:02<00:00, 71.8MB/s]


FasterRCNN


/tmp/ipykernel_10521/658116426.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## 6. Hàm eval COCO mAP trên tập validation


In [10]:
def evaluate_model(model, data_loader, device):
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        class_metrics=True,
    )

    model.eval()

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Evaluating", leave=False):
            images = [img.to(device) for img in images]
            outputs = model(images)

            preds = []
            gts = []

            for output, target in zip(outputs, targets):
                preds.append({
                    "boxes": output["boxes"].detach().cpu(),
                    "scores": output["scores"].detach().cpu(),
                    "labels": output["labels"].detach().cpu(),
                })
                gts.append({
                    "boxes": target["boxes"].detach().cpu(),
                    "labels": target["labels"].detach().cpu(),
                })

            metric.update(preds, gts)

    return metric.compute()


def tensor_to_python(x):
    if torch.is_tensor(x):
        if x.numel() == 1:
            return float(x.detach().cpu().item())
        return x.detach().cpu().tolist()
    return x


def save_eval_results(results, save_dir, filename_prefix="metrics_fasterrcnn"):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    summary = {
        "model": "Faster R-CNN ResNet50-FPN",
        "epochs": EPOCHS,
        "batch": BATCH,
        "imgsz": IMGSZ,
        "map_50_95": tensor_to_python(results["map"]),
        "map_50": tensor_to_python(results["map_50"]),
        "map_75": tensor_to_python(results["map_75"]),
        "map_small": tensor_to_python(results["map_small"]),
        "map_medium": tensor_to_python(results["map_medium"]),
        "map_large": tensor_to_python(results["map_large"]),
        "mar_1": tensor_to_python(results["mar_1"]),
        "mar_10": tensor_to_python(results["mar_10"]),
        "mar_100": tensor_to_python(results["mar_100"]),
    }

    summary_df = pd.DataFrame([summary])
    summary_csv = save_dir / f"{filename_prefix}_summary.csv"
    summary_json = save_dir / f"{filename_prefix}_summary.json"
    summary_df.to_csv(summary_csv, index=False)
    with open(summary_json, "w") as f:
        json.dump(summary, f, indent=2)

    per_class_rows = []
    classes = results.get("classes", torch.tensor([])).detach().cpu().tolist()
    map_per_class = results.get("map_per_class", torch.tensor([])).detach().cpu().tolist()
    mar_per_class = results.get("mar_100_per_class", torch.tensor([])).detach().cpu().tolist()

    for cls_id, ap, ar in zip(classes, map_per_class, mar_per_class):
        cls_id = int(cls_id)
        name_idx = cls_id - 1  # label 0 là background
        class_name = class_names[name_idx] if 0 <= name_idx < len(class_names) else f"class_{cls_id}"
        per_class_rows.append({
            "model": "Faster R-CNN ResNet50-FPN",
            "class_id_frcnn": cls_id,
            "class_id_yolo_equivalent": cls_id - 1,
            "class_name": class_name,
            "ap_50_95": float(ap),
            "ar_100": float(ar),
        })

    per_class_df = pd.DataFrame(per_class_rows)
    per_class_csv = save_dir / f"{filename_prefix}_per_class.csv"
    per_class_df.to_csv(per_class_csv, index=False)

    return summary_df, per_class_df, summary_csv, summary_json, per_class_csv


## 7. Train 30 epochs + eval trên val sau mỗi epoch

`best.pt` được lưu theo metric `mAP@0.50:0.95` cao nhất trên tập validation.


In [11]:
best_map = -1.0
history = []
last_ckpt_path = SAVE_DIR / "last.pt"
best_ckpt_path = SAVE_DIR / "best.pt"

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for images, targets in pbar:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = float(losses.detach().cpu().item())
        epoch_loss += loss_value
        pbar.set_postfix({
            "loss": f"{loss_value:.4f}",
            "lr": optimizer.param_groups[0]["lr"],
        })

    lr_scheduler.step()

    avg_train_loss = epoch_loss / max(1, len(train_loader))

    val_results = evaluate_model(model, val_loader, DEVICE)
    val_map = float(val_results["map"].detach().cpu().item())
    val_map50 = float(val_results["map_50"].detach().cpu().item())

    row = {
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_map_50_95": val_map,
        "val_map_50": val_map50,
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(SAVE_DIR / "training_history.csv", index=False)

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "lr_scheduler_state_dict": lr_scheduler.state_dict(),
        "class_names": class_names,
        "num_classes": num_classes,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "epochs": EPOCHS,
        "val_map_50_95": val_map,
        "val_map_50": val_map50,
    }

    torch.save(checkpoint, last_ckpt_path)

    if val_map > best_map:
        best_map = val_map
        torch.save(checkpoint, best_ckpt_path)
        print(f"Saved best.pt | epoch={epoch + 1} | mAP50-95={best_map:.4f}")

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"train_loss={avg_train_loss:.4f} "
        f"val_mAP50-95={val_map:.4f} "
        f"val_mAP50={val_map50:.4f}"
    )

print("Training completed.")
print("Best checkpoint:", best_ckpt_path)
print("Last checkpoint:", last_ckpt_path)


Epoch 1/30:   0%|          | 0/175 [00:00<?, ?it/s]

/tmp/ipykernel_10521/1606334102.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Saved best.pt | epoch=1 | mAP50-95=0.0160
Epoch [1/30] train_loss=0.7003 val_mAP50-95=0.0160 val_mAP50=0.0513


Epoch 2/30:   0%|          | 0/175 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch [29/30] train_loss=0.2135 val_mAP50-95=0.1202 val_mAP50=0.2976


Epoch 30/30:   0%|          | 0/175 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch [30/30] train_loss=0.2132 val_mAP50-95=0.1214 val_mAP50=0.2977
Training completed.
Best checkpoint: /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/best.pt
Last checkpoint: /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/last.pt


## 8. Load `best.pt` và eval cuối cùng trên cùng tập validation


In [12]:
checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

final_results = evaluate_model(model, val_loader, DEVICE)

summary_df, per_class_df, summary_csv, summary_json, per_class_csv = save_eval_results(
    final_results,
    SAVE_DIR,
    filename_prefix="metrics_fasterrcnn",
)

print("===== Faster R-CNN Overall Metrics =====")
display(summary_df)

print("===== Faster R-CNN Per-class Metrics =====")
display(per_class_df)

print("Saved summary CSV:", summary_csv)
print("Saved summary JSON:", summary_json)
print("Saved per-class CSV:", per_class_csv)


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

===== Faster R-CNN Overall Metrics =====


,model,epochs,batch,imgsz,map_50_95,map_50,map_75,map_small,map_medium,map_large,mar_1,mar_10,mar_100
0,Faster R-CNN ResNet50-FPN,30,16,640,0.127722,0.305678,0.084241,0.000179,0.006529,0.144076,0.186513,0.32329,0.32757


===== Faster R-CNN Per-class Metrics =====


,model,class_id_frcnn,class_id_yolo_equivalent,class_name,ap_50_95,ar_100
0,Faster R-CNN ResNet50-FPN,1,0,crack,0.120418,0.379141
1,Faster R-CNN ResNet50-FPN,2,1,dent,0.119089,0.331911
2,Faster R-CNN ResNet50-FPN,3,2,glass shatter,0.061094,0.228346
3,Faster R-CNN ResNet50-FPN,4,3,lamp broken,0.104036,0.282836
4,Faster R-CNN ResNet50-FPN,5,4,scratch,0.160467,0.367473
5,Faster R-CNN ResNet50-FPN,6,5,tire flat,0.201228,0.375714


Saved summary CSV: /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/metrics_fasterrcnn_summary.csv
Saved summary JSON: /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/metrics_fasterrcnn_summary.json
Saved per-class CSV: /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/metrics_fasterrcnn_per_class.csv


## 9. In nhanh đường dẫn output


In [13]:
print("Output dir       :", SAVE_DIR)
print("Best checkpoint  :", best_ckpt_path)
print("Last checkpoint  :", last_ckpt_path)
print("Training history :", SAVE_DIR / "training_history.csv")
print("Summary CSV      :", summary_csv)
print("Per-class CSV    :", per_class_csv)


Output dir       : /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640
Best checkpoint  : /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/best.pt
Last checkpoint  : /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/last.pt
Training history : /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/training_history.csv
Summary CSV      : /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/metrics_fasterrcnn_summary.csv
Per-class CSV    : /content/drive/MyDrive/coco_damage_car_yolov8/fasterrcnn_car_30e_640/metrics_fasterrcnn_per_class.csv
